
# SILVER LAYER - FHIR Encounter Processing (Production Ready)



In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime
import logging
import json
from pyspark.sql.functions import (
    col, trim, lower, row_number
)
from pyspark.sql.window import Window

In [0]:
# ====================== Config Details ======================
CONFIG = {
    "resource": "Encounter",
    "run_date": datetime.today().strftime("%Y-%m-%d"),
    "bronze_path": "fhir_data.prd_bronze",
    "silver_path": "fhir_data.prd_silver"
}

resource = CONFIG["resource"]
run_date = CONFIG["run_date"]
bronze_base = CONFIG["bronze_path"]
silver_base = CONFIG["silver_path"]

In [0]:
# ====================== LOGGING ======================
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)

logger.info(f"Starting Silver Layer | Resource: {resource} | Run Date: {run_date}")

In [0]:
# ====================== SCHEMA ======================
from pyspark.sql.types import *

schema = StructType([
    StructField("id", StringType(), True),

    StructField("identifier", ArrayType(
        StructType([
            StructField("system", StringType(), True),
            StructField("value", StringType(), True)
        ])
    ), True),

    StructField("meta", StructType([
        StructField("lastUpdated", StringType(), True),
        StructField("source", StringType(), True),
        StructField("versionId", StringType(), True)
    ]), True),

    StructField("resourceType", StringType(), True),

    StructField("serviceProvider", StructType([
        StructField("reference", StringType(), True)
    ]), True),

    StructField("status", StringType(), True),

    StructField("subject", StructType([
        StructField("reference", StringType(), True)
    ]), True)
])

# ====================== MAIN FUNCTION ======================
def process_patient_silver(resource, run_date, bronze_base):

    bronze_path = f"{bronze_base}.{resource.lower()}"

    logger.info(f"Reading bronze data from: {bronze_path}")

    df = spark.read\
        .table(bronze_path) \
        .filter(col("load_date") == run_date)
    df.display()

    logger.info(f"Initial count: {df.count()}")

    df = df.dropDuplicates()
# ================= PARSE =================
    df_parsed = df.withColumn("parsed", from_json(col("resource"), schema))
#======================== Flattening the Data ============================
    df_silver = (
    df_parsed
    # explode identifier array from parsed object
    .withColumn("identifier", explode_outer(col("parsed.identifier")))

    # flatten parsed object into columns
    .select(
        col("parsed.id").alias("encounter_id"),

        col("identifier.system").alias("identifier_system"),
        col("identifier.value").alias("identifier_value"),

        col("parsed.resourceType").alias("resource_type"),
        col("parsed.status").alias("status"),

        split(col("parsed.subject.reference"), "/")[1].alias("patient_id"),
        split(col("parsed.serviceProvider.reference"), "/")[1].alias("organization_id"),

        to_timestamp(col("parsed.meta.lastUpdated")).alias("last_updated_ts"),
        col("parsed.meta.source").alias("source"),
        col("parsed.meta.versionId").alias("version_id"),

        current_date().alias("ingestion_date")
    ))

    #df_silver.display()
    # Step 1: Basic column cleanup (string hygiene)
    df_clean = (
    df_silver
    .withColumn("encounter_id", trim(col("encounter_id")))
    .withColumn("patient_id", trim(col("patient_id")))
    .withColumn("organization_id", trim(col("organization_id")))
    .withColumn("status", lower(trim(col("status"))))
    .withColumn("resource_type", trim(col("resource_type"))))
    df_silver.display()
    #Step 2: Remove corrupt / invalid records
    df_valid = (
    df_clean
    .withColumn(
        "identifier_type",
        when(col("identifier_value").rlike("^[0-9]+$"), "NUMERIC")
        .when(col("identifier_value").rlike("^INP_"), "INPATIENT")
        .otherwise("UNKNOWN")
    )
    .withColumn(
        "identifier_numeric",
        regexp_extract(col("identifier_value"), "([0-9]+)", 1)
    ))
    df_valid = (
    df_valid
    #.filter(col("identifier_type") != "UNKNOWN")
    .filter(col("encounter_id").isNotNull())
    .filter(col("patient_id").isNotNull())
    .filter(col("status").isNotNull())
    .filter(col("last_updated_ts").isNotNull())
    .filter(col("resource_type") == "Encounter")
    .drop('identifier_value'))

    valid_status = ["finished", "in-progress", "planned", "cancelled"]
    df_valid = df_valid.filter(col("status").isin(valid_status))
    window_spec = Window.partitionBy("encounter_id").orderBy(
    col("last_updated_ts").desc())
    df_dedup = (
    df_valid
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn"))
    df_dedup.display()
    return df_dedup

In [0]:

# ====================== Main EXECUTION ======================
try:
    df_silver = process_patient_silver(resource, run_date, bronze_base)
    df_silver.createOrReplaceTempView("silver_tmp")

    silver_path = f"{silver_base}.{resource.lower()}"
    spark.sql(f"""
MERGE INTO {silver_path} t
USING silver_tmp s
ON t.encounter_id = s.encounter_id

WHEN MATCHED THEN UPDATE SET
    t.encounter_id = s.encounter_id,
    t.identifier_system = s.identifier_system,
    t.resource_type = s.resource_type,
    t.status = s.status,
    t.patient_id = s.patient_id,
    t.organization_id = s.organization_id,
    t.last_updated_ts = current_timestamp(),
    t.source = s.source,
    t.version_id = s.version_id,
    t.ingestion_date = s.ingestion_date,
    t.identifier_type = s.identifier_type,
    t.identifier_numeric = s.identifier_numeric

WHEN NOT MATCHED THEN INSERT (
    encounter_id,
    identifier_system,
    resource_type,
    status,
    patient_id,
    organization_id,
    last_updated_ts,
    source,
    version_id,
    ingestion_date,
    identifier_type,
    identifier_numeric
)
VALUES (
    s.encounter_id,
    s.identifier_system,
    s.resource_type,
    s.status,
    s.patient_id,
    s.organization_id,
    current_timestamp(),
    s.source,
    s.version_id,
    s.ingestion_date,
    s.identifier_type,
    s.identifier_numeric
)
""")

    # df_silver.write.format("delta") \
    #     .mode("append") \
    #     .option("mergeSchema", "true") \
    #     .saveAsTable(silver_path)

    logger.info(f"Successfully written to Silver: {silver_path}")

    #df_silver.display()
    df_silver.printSchema()

except Exception as e:
    logger.error(f"Pipeline failed: {str(e)}", exc_info=True)
    raise

finally:
    logger.info("Pipeline execution completed")